# 02 — Select and audit the conservative international cohort

This notebook **reuses completed global cleaning**; it does not perform a second, potentially inconsistent missing-value/outlier policy. It selects only explicit named non-U.S. locations, verifies IDs and schema, records geographic provenance, and writes a separate CSV under `data/Processed-Data/international_only/`.

Never relabel the entire complement of the U.S. state-code mask as international. **No global or U.S. artifact is overwritten.**

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd()
PROJECT_ROOT = next((folder for folder in (HERE, *HERE.parents)
                     if (folder / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the 1C repository")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
GLOBAL_DIR = PROJECT_ROOT / "data" / "Processed-Data"
INT_DIR = GLOBAL_DIR / "international_only"
CLEAN_GLOBAL = GLOBAL_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
import pandas as pd
from utils.international_scope import classify_locations, international_mask, location_scope_audit
if not CLEAN_GLOBAL.is_file():
    raise FileNotFoundError(f"Missing global cleaned CSV: {CLEAN_GLOBAL}")
global_clean = pd.read_csv(CLEAN_GLOBAL)
assert global_clean["project id"].is_unique
geo, scope_table, countries, uncertain = location_scope_audit(global_clean)
international = global_clean.loc[geo["scope"].eq("explicit_non_us")].copy()
provenance = geo.loc[geo["scope"].eq("explicit_non_us")].copy()
provenance.insert(0, "project id", international["project id"].to_numpy())
assert len(international) == len(provenance)
assert international["project id"].is_unique
assert international_mask(international).all()
assert international.columns.tolist() == global_clean.columns.tolist()
assert international["project id"].isin(global_clean["project id"]).all()
print("All-status explicitly named international projects:", len(international))
print("Settled (successful/failed):", int(international["status"].isin(["successful", "failed"]).sum()))
print("Unknown and ambiguous projects excluded:", int(geo["scope"].eq("unknown").sum()))
display(scope_table)
display(provenance["country_region"].value_counts().head(25).to_frame("n"))

All-status explicitly named international projects: 2279
Settled (successful/failed): 2064
Unknown and ambiguous projects excluded: 1333


,scope,evidence,n
0,explicit_non_us,explicit_country_or_region_name,2279
1,explicit_us,explicit_us_name,2
2,unknown,ambiguous_country_name,8
3,unknown,missing_location,1318
4,unknown,no_explicit_country_name,7
5,us_candidate,us_state_suffix,42175
6,us_territory,us_territory_excluded,26


,n
country_region,
Canada,284
United Kingdom,216
Germany,119
Mexico,84
France,81
Italy,74
India,68
China,60
Australia,57


In [2]:
# Preserve the original cleaned 17-column schema; provenance stays separately auditable.
INT_DIR.mkdir(exist_ok=True, parents=True)
clean_dest = INT_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
international.to_csv(clean_dest, index=False)
provenance.to_csv(INT_DIR / "country_region_provenance.csv", index=False)
scope_table.to_csv(INT_DIR / "location_scope_summary.csv", index=False)
countries.to_csv(INT_DIR / "named_country_counts.csv", index=False)
# Save unknown-location *frequencies* only for manual review; never include them in this cohort.
uncertain.to_csv(INT_DIR / "unknown_location_counts.csv", index=False)
reloaded = pd.read_csv(clean_dest)
assert reloaded["project id"].is_unique
assert len(reloaded) == len(international)
assert international_mask(reloaded).all()
print("Saved international-only files to:", INT_DIR)
print("Did not modify:", CLEAN_GLOBAL)

Saved international-only files to: /Users/annasehgal/State-Street-1C-kickstarter-crowdfunding-recommendation-engine/data/Processed-Data/international_only
Did not modify: /Users/annasehgal/State-Street-1C-kickstarter-crowdfunding-recommendation-engine/data/Processed-Data/DSI_kickstarterscrape_dataset_cleaned.csv


**Next:** `03_preprocess_data.ipynb` retains successful/failed campaigns and intersects their IDs with the **existing global official train/test split**. A new random split would compromise comparability with the already-reserved U.S. holdout. Unknown-location and territory records remain excluded unless a separate, verified evidence source is added.